# 01 – Cleaning the DBAASP peptide data

**Goal:** turn the raw `by_origin/*.csv` files into one clean table that the later notebooks can use for comparing groups.

**Why `by_origin`?** Every peptide is in exactly one of `Natural.csv` / `Synthetic.csv`, so stacking them rebuilds the full master table. The other folders (`by_kingdom`, `by_length_class`, ...) are re-slices of the same peptides, so we do not need to load them.

**About the data source.** The records come from **DBAASP**, a curated database of peptides that were tested for antimicrobial and cytotoxic activity (Pirtskhalava et al. 2016, 2021 – see *References* at the end). Two facts from those papers drive the cleaning below:

* DBAASP stores three kinds of records: *monomers*, *homo-/hetero-multimers* and *multi-peptides* (Pirtskhalava et al. 2021).
* In a DBAASP sequence, **upper-case** letters are L-amino acids (or amino acids without a stereoisomer), **lower-case** letters are **D-stereoisomers**, and an **`X` or `x`** marks an *unusual or post-translationally modified amino acid* (Pirtskhalava et al. 2016, "Information about chemical structure").

**Cleaning steps** (each one logs how many rows it removes):
1. Load and stack Natural + Synthetic
2. Remove exact duplicate rows
3. Keep monomers only
4. Fix the sequence column (D-amino acids, `X` residues)
5. Turn the multi-label `TARGET GROUP` column into yes/no columns and save

In [1]:
import ast
import pandas as pd
import matplotlib.pyplot as plt

DATA = ".."          # repo root, relative to this notebook
OUT = "../results"

# We record (step name, rows left) after every step -> becomes the cleaning log
log = []
def record(step, df):
    origin = df["ORIGIN"] if "ORIGIN" in df else df["origin"]   # column is renamed at the end
    log.append({"step": step, "rows": len(df),
                "natural": int((origin == "Natural").sum()),
                "synthetic": int((origin == "Synthetic").sum())})
    print(f"{step:<45} rows = {len(df):>6}")

## 1. Load and stack Natural + Synthetic

In [2]:
raw = pd.concat(
    [pd.read_csv(f"{DATA}/by_origin/Natural.csv"),
     pd.read_csv(f"{DATA}/by_origin/Synthetic.csv")],
    ignore_index=True,
)
record("1. raw (Natural + Synthetic)", raw)
raw.head(3)

1. raw (Natural + Synthetic)                  rows =  25542


,ID,COMPLEXITY,NAME,N TERMINUS,SEQUENCE,C TERMINUS,SYNTHESIS TYPE,TARGET GROUP,TARGET OBJECT,tokenised_TARGET GROUP,tokenised_TARGET OBJECT,ORIGIN,Length_of_Sequence,Length_Class,KINGDOM,DOMAIN
0,1,Multimer,Distinctin,NaN,NLVSGLIEARKYLEQLHRKLKNCKV ENREVPPGFTALIKTLR...,NaN,Ribosomal,Gram+ Gram-,Lipid Bilayer,"['Gram+', 'Gram-']",['Lipid Bilayer'],Natural,51,Long (50–100 aa),Unclassified,NaN
1,3,Multimer,Halocidin,NaN,WLNALLHHGLNCAKGVLA ALLHHGLNCAKGVLA,AMD AMD,Ribosomal,Gram+,Lipid Bilayer,['Gram+'],['Lipid Bilayer'],Natural,37,Medium (25–50 aa),Unclassified,NaN
2,6,Multi-Peptide,Enterocin X,NaN,SNDSLWYGVGQFMGKQANCITNHPVKHMIIPGYCLSKILG IA...,NaN,Ribosomal,Gram+,Lipid Bilayer,['Gram+'],['Lipid Bilayer'],Natural,81,Long (50–100 aa),Unclassified,NaN


## 2. Remove exact duplicate rows
The raw stack has more rows than unique IDs. DBAASP gives every chemically distinct peptide its own record (Pirtskhalava et al. 2016), so a row that is repeated word for word cannot be a different peptide. We check that the extra rows are exact copies and then drop them.

In [3]:
print("rows:", len(raw), "| unique IDs:", raw["ID"].nunique(), "| exact duplicate rows:", raw.duplicated().sum())

df = raw.drop_duplicates().copy()
assert df["ID"].is_unique, "some IDs still repeat with different content"
record("2. drop exact duplicates", df)

rows: 25542 | unique IDs: 23542 | exact duplicate rows: 2000
2. drop exact duplicates                      rows =  23542


## 3. Keep monomers only
`COMPLEXITY` is Monomer, Multimer or Multi-Peptide (Pirtskhalava et al. 2021). Multimers and multi-peptides store several chains in one sequence cell (separated by spaces), so a per-peptide value such as net charge would mix chains. DBAASP's own structure-modelling pipeline is also restricted to monomers (Pirtskhalava et al. 2021, "MD modelling").

In [4]:
print(df["COMPLEXITY"].value_counts())
df = df[df["COMPLEXITY"] == "Monomer"].copy()
record("3. keep monomers only", df)

COMPLEXITY
Monomer          22907
Multimer           405
Multi-Peptide      230
Name: count, dtype: int64
3. keep monomers only                         rows =  22907


## 4. Fix the sequence column
Using the DBAASP conventions described above:

* **Lower-case letters (D-amino acids):** we change them to upper case so the amino acid is counted, and keep a flag `has_D_aa`.
* **`X` / `x` (unusual or modified amino acids):** these positions cannot be assigned to one of the 20 standard amino acids, so charge, hydropathy and composition are undefined for them (the Kyte-Doolittle scale used in notebook 03 only has the 20 standard amino acids). Sequences containing them are removed.

In [5]:
df["has_D_aa"] = df["SEQUENCE"].str.contains("[a-z]")
df["SEQUENCE"] = df["SEQUENCE"].str.upper()

standard = df["SEQUENCE"].str.fullmatch("[ACDEFGHIKLMNPQRSTVWY]+")
print("sequences with an X / non-standard residue:", (~standard).sum())
print("share removed by origin (should be similar, otherwise we bias the comparison):")
print((~standard).groupby(df["ORIGIN"]).mean().round(3))

df = df[standard].copy()
assert (df["Length_of_Sequence"] == df["SEQUENCE"].str.len()).all(), "length column disagrees with sequence"
record("4. drop sequences with X / non-standard", df)

sequences with an X / non-standard residue: 4632
share removed by origin (should be similar, otherwise we bias the comparison):
ORIGIN
Natural      0.188
Synthetic    0.205
Name: SEQUENCE, dtype: float64
4. drop sequences with X / non-standard       rows =  18275


## 5. Turn `TARGET GROUP` into yes/no columns and save
`TARGET GROUP` is multi-label (e.g. `Gram+ Gram- Cancer`). We make one 0/1 column per target group (`target_...`).

**Wording matters here.** DBAASP records peptides that were tested experimentally *"regardless of whether the peptides are determined to be active or inactive"* (Pirtskhalava et al. 2021). We have not been able to check how the `TARGET GROUP` field treats inactive peptides, so we say a peptide is **"listed with"** a target group, and we do **not** call it "active against" that group.

Rows with no target information stay in, but get `has_target_info = False` so they can be left out of target comparisons.

In [6]:
df["targets"] = df["tokenised_TARGET GROUP"].apply(ast.literal_eval)

target_cols = {
    "Gram+": "target_gram_pos", "Gram-": "target_gram_neg", "Fungus": "target_fungus",
    "Cancer": "target_cancer", "Virus": "target_virus", "Biofilm": "target_biofilm",
    "Parasite": "target_parasite", "Insect": "target_insect", "Nematode": "target_nematode",
    "Mollicute": "target_mollicute", "Protista": "target_protista", "Archaea": "target_archaea",
    "Mammalian Cell": "target_mammalian_cell",
}
for label, col in target_cols.items():
    df[col] = df["targets"].apply(lambda t: int(label in t))

df["has_target_info"] = df["targets"].str.len() > 0
print("rows without target info:", (~df["has_target_info"]).sum())

keep = (["ID", "NAME", "SEQUENCE", "Length_of_Sequence", "Length_Class", "ORIGIN", "SYNTHESIS TYPE",
         "has_D_aa", "has_target_info"] + list(target_cols.values()))
clean = df[keep].rename(columns={
    "SEQUENCE": "sequence", "Length_of_Sequence": "length", "Length_Class": "length_class",
    "ORIGIN": "origin", "SYNTHESIS TYPE": "synthesis_type", "NAME": "name",
}).reset_index(drop=True)

# --- sanity checks: the notebook stops here if something is off ---
assert clean["ID"].is_unique
assert clean["sequence"].str.fullmatch("[ACDEFGHIKLMNPQRSTVWY]+").all()
assert (clean["length"] == clean["sequence"].str.len()).all()
assert set(clean["origin"]) == {"Natural", "Synthetic"}
record("5. add target columns (final)", clean)
clean[list(target_cols.values())].sum().sort_values(ascending=False)

rows without target info: 329
5. add target columns (final)                 rows =  18275


target_gram_neg          14829
target_gram_pos          14217
target_mammalian_cell    10326
target_fungus             5357
target_cancer             3143
target_virus              1252
target_biofilm             613
target_parasite            322
target_insect               96
target_nematode             44
target_protista             44
target_mollicute            34
target_archaea               2
dtype: int64

## Cleaning log

In [7]:
log_df = pd.DataFrame(log)
log_df["removed"] = (log_df["rows"].shift(1) - log_df["rows"]).fillna(0).astype(int)
log_df

,step,rows,natural,synthetic,removed
0,1. raw (Natural + Synthetic),25542,4493,21049,0
1,2. drop exact duplicates,23542,4087,19455,2000
2,3. keep monomers only,22907,3990,18917,635
3,4. drop sequences with X / non-standard,18275,3239,15036,4632
4,5. add target columns (final),18275,3239,15036,0


In [8]:
fig, ax = plt.subplots(figsize=(8, 3.5))
labels = [s.split(". ", 1)[1] for s in log_df["step"]]
ax.barh(labels[::-1], log_df["natural"][::-1], label="Natural")
ax.barh(labels[::-1], log_df["synthetic"][::-1], left=log_df["natural"][::-1], label="Synthetic")
ax.set_xlabel("rows remaining")
ax.set_title("Rows remaining after each cleaning step")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/cleaning_funnel.png", dpi=150)
plt.show()

## Quick look at the final data

In [9]:
print(clean["origin"].value_counts(), "\n")
print(clean.groupby("origin")["length"].describe().round(1), "\n")
print("has_D_aa by origin:\n", clean.groupby("origin")["has_D_aa"].mean().round(3), "\n")
print("has_target_info by origin:\n", clean.groupby("origin")["has_target_info"].mean().round(3))

origin
Synthetic    15036
Natural       3239
Name: count, dtype: int64 

             count  mean   std  min   25%   50%   75%    max
origin                                                      
Natural     3239.0  27.2  16.5  2.0  17.0  25.0  35.0  190.0
Synthetic  15036.0  17.1   9.4  1.0  11.0  15.0  21.0  148.0 

has_D_aa by origin:
 origin
Natural      0.035
Synthetic    0.100
Name: has_D_aa, dtype: float64 

has_target_info by origin:
 origin
Natural      0.992
Synthetic    0.980
Name: has_target_info, dtype: float64


## Save

In [10]:
clean.to_csv(f"{OUT}/clean_peptides.csv", index=False)
log_df.to_csv(f"{OUT}/cleaning_log.csv", index=False)
print("saved", clean.shape, "->", f"{OUT}/clean_peptides.csv")

saved (18275, 22) -> ../results/clean_peptides.csv


## Notes for the methods / limitations section
* Exact duplicate rows were removed; IDs are unique afterwards. Different IDs can still hold the same or a very similar sequence; that is handled in notebook 02.
* Multimers and multi-peptides were excluded (sequence features are only defined for a single chain).
* D-amino acids were converted to their upper-case letter (flag kept in `has_D_aa`).
* Sequences with `X` / `x` were dropped. The share removed was similar for both origins (see step 4), so this should not favour one group.
* "Natural" and "Synthetic" come from the `ORIGIN` column of the repository files (Natural = ribosomal + non-ribosomal synthesis types, Synthetic = synthetic).
* The **target-group columns overlap** (a peptide can be listed with Gram+ and Gram- and Cancer). Later comparisons must not treat them as independent groups.
* A target group means the peptide is *listed* with it in DBAASP, not necessarily that it is active against it (see step 5).

## References
* Pirtskhalava M, et al. (2016) DBAASP v.2: an enhanced database of structure and antimicrobial/cytotoxic activity of natural and synthetic peptides. *Nucleic Acids Res* 44(D1):D1104-D1112. https://doi.org/10.1093/nar/gkv1174 (PMID 26578581)
* Pirtskhalava M, et al. (2021) DBAASP v3: database of antimicrobial/cytotoxic activity and structure of peptides as a resource for development of new therapeutics. *Nucleic Acids Res* 49(D1):D288-D297. https://doi.org/10.1093/nar/gkaa991 (PMID 33151284)